# VR Controller Evaluation Analysis

This notebook analyzes the 10 Quest CSV logs stored in `../VRData`.

It is structured to support the paper's technical evaluation:

- **H1 (Accumulation):** under sustained low head rotation, `Ladapted` should show a predominantly non-decreasing trend while remaining bounded.
- **H2 (Safety bounding):** under rapid head rotation, the controller should increase discomfort, lower the multiplier toward its minimum, and damp `Ladapted` relative to `L`.
- **H3 (Stability):** the controller should remain bounded and avoid sustained oscillatory instability.

All metrics are computed on the **active phase only**, defined as rows where `ActiveTriggers > 0`.


In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'VRData'
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

LOW_VELOCITY_THRESHOLD = 30.0
HIGH_VELOCITY_THRESHOLD = 90.0
MIN_MULTIPLIER = 0.8
MAX_MULTIPLIER = 1.2

STILL_FILES = [
    'adaptation_data_20260317_134300.csv',
    'adaptation_data_20260317_134357.csv',
    'adaptation_data_20260317_134519.csv',
    'adaptation_data_20260317_134619.csv',
    'adaptation_data_20260317_134723.csv',
]

RAPID_FILES = [
    'adaptation_data_20260317_134832.csv',
    'adaptation_data_20260317_135526.csv',
    'adaptation_data_20260317_135618.csv',
    'adaptation_data_20260317_135751.csv',
    'adaptation_data_20260317_135904.csv',
]

print('Data directory:', DATA_DIR)
print('Output directory:', OUTPUT_DIR.resolve())


Data directory: /Users/frederik/Desktop/kode/aau/med7/med7/VRData
Output directory: /Users/frederik/Desktop/kode/aau/med7/med7/analysis/output


In [8]:
def load_active_phase(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    active = df[df['ActiveTriggers'] > 0].copy()
    if active.empty:
        active = df.copy()
    active = active.reset_index(drop=True)
    active['Time'] = active['Time'] - active['Time'].iloc[0]
    return active


def derivative_ratio(values: pd.Series) -> float:
    diffs = values.diff().iloc[1:]
    if diffs.empty:
        return float('nan')
    return float((diffs >= -1e-6).mean())


def max_sign_changes_per_window(values: pd.Series, times: pd.Series, window_seconds: float = 1.0) -> int:
    diffs = values.diff().iloc[1:]
    if diffs.empty:
        return 0

    slope_times = times.iloc[1:].to_numpy(dtype=float)
    slopes = diffs.to_numpy(dtype=float)

    def sign(v: float, eps: float = 1e-6) -> int:
        if v > eps:
            return 1
        if v < -eps:
            return -1
        return 0

    signs = [sign(v) for v in slopes]
    max_changes = 0
    for start in range(len(signs)):
        changes = 0
        last = signs[start]
        for current in range(start + 1, len(signs)):
            if slope_times[current] - slope_times[start] > window_seconds:
                break
            if signs[current] != 0 and last != 0 and signs[current] != last:
                changes += 1
            if signs[current] != 0:
                last = signs[current]
        max_changes = max(max_changes, changes)
    return max_changes


def summarize_run(file_name: str, scenario: str, df: pd.DataFrame) -> dict:
    high_velocity = df['HeadRotationSpeed'] >= HIGH_VELOCITY_THRESHOLD
    return {
        'scenario': scenario,
        'run': file_name,
        'samples_active_phase': int(len(df)),
        'duration_s': float(df['Time'].iloc[-1]) if len(df) > 0 else 0.0,
        'mean_head_rotation_speed': float(df['HeadRotationSpeed'].mean()),
        'max_head_rotation_speed': float(df['HeadRotationSpeed'].max()),
        'mean_adaptation_multiplier': float(df['AdaptationMultiplier'].mean()),
        'min_adaptation_multiplier': float(df['AdaptationMultiplier'].min()),
        'max_adaptation_multiplier': float(df['AdaptationMultiplier'].max()),
        'mean_adapted_level': float(df['AdaptedLevel'].mean()),
        'max_adapted_level': float(df['AdaptedLevel'].max()),
        'mean_discomfort': float(df['DiscomfortLevel'].mean()),
        'max_discomfort': float(df['DiscomfortLevel'].max()),
        'high_velocity_ratio': float(high_velocity.mean()),
        'overstimulation_bound_violations': int((~df['OverstimulationLevel'].between(0.0, 1.0)).sum()),
        'adapted_bound_violations': int((~df['AdaptedLevel'].between(0.0, 1.0)).sum()),
        'multiplier_bound_violations': int((~df['AdaptationMultiplier'].between(MIN_MULTIPLIER, MAX_MULTIPLIER)).sum()),
        'non_negative_adapted_derivative_ratio': derivative_ratio(df['AdaptedLevel']),
        'adapted_below_overstimulation_ratio': float((df['AdaptedLevel'] < df['OverstimulationLevel']).mean()),
        'adapted_below_overstimulation_ratio_rapid_regime': float((df.loc[high_velocity, 'AdaptedLevel'] < df.loc[high_velocity, 'OverstimulationLevel']).mean()) if high_velocity.any() else np.nan,
        'max_sign_changes_1s': int(max_sign_changes_per_window(df['AdaptedLevel'], df['Time'])),
    }


def load_runs(file_names, scenario_name):
    runs = []
    summaries = []
    for name in file_names:
        df = load_active_phase(DATA_DIR / name)
        runs.append((name, df))
        summaries.append(summarize_run(name, scenario_name, df))
    return runs, pd.DataFrame(summaries)


def interpolate_mean_curves(runs, num_points=400):
    max_time = max(df['Time'].iloc[-1] for _, df in runs)
    shared_time = np.linspace(0.0, max_time, num_points)
    overstimulation_curves = []
    adapted_curves = []
    multiplier_curves = []

    for _, df in runs:
        t = df['Time'].to_numpy(dtype=float)
        overstimulation_curves.append(np.interp(shared_time, t, df['OverstimulationLevel'].to_numpy(dtype=float), left=df['OverstimulationLevel'].iloc[0], right=df['OverstimulationLevel'].iloc[-1]))
        adapted_curves.append(np.interp(shared_time, t, df['AdaptedLevel'].to_numpy(dtype=float), left=df['AdaptedLevel'].iloc[0], right=df['AdaptedLevel'].iloc[-1]))
        multiplier_curves.append(np.interp(shared_time, t, df['AdaptationMultiplier'].to_numpy(dtype=float), left=df['AdaptationMultiplier'].iloc[0], right=df['AdaptationMultiplier'].iloc[-1]))

    return pd.DataFrame({
        'Time': shared_time,
        'MeanOverstimulationLevel': np.mean(overstimulation_curves, axis=0),
        'MeanAdaptedLevel': np.mean(adapted_curves, axis=0),
        'MeanAdaptationMultiplier': np.mean(multiplier_curves, axis=0),
    })


def make_mean_plot(mean_df: pd.DataFrame, title: str, output_path: Path):
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.plot(mean_df['Time'], mean_df['MeanOverstimulationLevel'], label='Mean L', linewidth=2.6, color='#1f4e79')
    ax.plot(mean_df['Time'], mean_df['MeanAdaptedLevel'], label='Mean Ladapted', linewidth=2.6, color='#c55a11')
    ax.set_xlabel('Time since active phase start (s)')
    ax.set_ylabel('Level')
    ax.set_ylim(0.0, 1.05)
    ax.set_title(title)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    plt.close(fig)


In [9]:
still_runs, still_summary = load_runs(STILL_FILES, 'still')
rapid_runs, rapid_summary = load_runs(RAPID_FILES, 'rapid')

per_run_summary = pd.concat([still_summary, rapid_summary], ignore_index=True)
per_run_summary.to_csv(OUTPUT_DIR / 'per_run_summary.csv', index=False)

per_run_summary


,scenario,run,samples_active_phase,duration_s,mean_head_rotation_speed,max_head_rotation_speed,mean_adaptation_multiplier,min_adaptation_multiplier,max_adaptation_multiplier,mean_adapted_level,...,mean_discomfort,max_discomfort,high_velocity_ratio,overstimulation_bound_violations,adapted_bound_violations,multiplier_bound_violations,non_negative_adapted_derivative_ratio,adapted_below_overstimulation_ratio,adapted_below_overstimulation_ratio_rapid_regime,max_sign_changes_1s
0,still,adaptation_data_20260317_134300.csv,212,23.419,0.235425,2.70,1.200000,1.2,1.2000,0.793346,...,0.000000,0.0,0.000000,0,0,0,1.000000,0.000000,NaN,0
1,still,adaptation_data_20260317_134357.csv,237,26.157,1.211561,18.96,1.200000,1.2,1.2000,0.812148,...,0.000000,0.0,0.000000,0,0,0,1.000000,0.000000,NaN,0
2,still,adaptation_data_20260317_134519.csv,244,26.814,0.713115,3.66,1.200000,1.2,1.2000,0.820277,...,0.000000,0.0,0.000000,0,0,0,1.000000,0.000000,NaN,0
3,still,adaptation_data_20260317_134619.csv,294,32.469,0.435238,4.91,1.200000,1.2,1.2000,0.848521,...,0.000000,0.0,0.000000,0,0,0,1.000000,0.000000,NaN,0
4,still,adaptation_data_20260317_134723.csv,244,27.070,1.920451,16.40,1.200000,1.2,1.2000,0.823262,...,0.000000,0.0,0.000000,0,0,0,1.000000,0.000000,NaN,0
5,rapid,adaptation_data_20260317_134832.csv,251,27.369,137.815179,215.42,0.867509,0.8,1.1915,0.639872,...,0.853161,1.0,0.780876,0,0,0,0.828000,0.856574,0.979592,1
6,rapid,adaptation_data_20260317_135526.csv,234,25.305,112.142308,188.99,0.887796,0.8,1.2000,0.629699,...,0.802537,1.0,0.769231,0,0,0,0.939914,0.799145,0.977778,1
7,rapid,adaptation_data_20260317_135618.csv,236,25.622,143.181822,240.78,0.897775,0.8,1.2000,0.636214,...,0.781259,1.0,0.741525,0,0,0,0.927660,0.766949,0.982857,1
8,rapid,adaptation_data_20260317_135751.csv,268,29.198,141.155634,215.34,0.882549,0.8,1.2000,0.653062,...,0.822604,1.0,0.798507,0,0,0,0.940075,0.805970,0.981308,1
9,rapid,adaptation_data_20260317_135904.csv,308,33.683,148.460747,219.51,0.856797,0.8,1.2000,0.664097,...,0.887333,1.0,0.873377,0,0,0,0.990228,0.866883,0.992565,2


In [10]:
scenario_summary = per_run_summary.groupby('scenario', as_index=False).agg({
    'samples_active_phase': 'mean',
    'duration_s': 'mean',
    'mean_head_rotation_speed': 'mean',
    'max_head_rotation_speed': 'mean',
    'mean_adaptation_multiplier': 'mean',
    'min_adaptation_multiplier': 'mean',
    'max_adaptation_multiplier': 'mean',
    'mean_adapted_level': 'mean',
    'max_adapted_level': 'mean',
    'mean_discomfort': 'mean',
    'max_discomfort': 'mean',
    'high_velocity_ratio': 'mean',
    'overstimulation_bound_violations': 'sum',
    'adapted_bound_violations': 'sum',
    'multiplier_bound_violations': 'sum',
    'non_negative_adapted_derivative_ratio': 'mean',
    'adapted_below_overstimulation_ratio': 'mean',
    'adapted_below_overstimulation_ratio_rapid_regime': 'mean',
    'max_sign_changes_1s': 'max',
})

scenario_summary.to_csv(OUTPUT_DIR / 'scenario_summary.csv', index=False)
scenario_summary


,scenario,samples_active_phase,duration_s,mean_head_rotation_speed,max_head_rotation_speed,mean_adaptation_multiplier,min_adaptation_multiplier,max_adaptation_multiplier,mean_adapted_level,max_adapted_level,mean_discomfort,max_discomfort,high_velocity_ratio,overstimulation_bound_violations,adapted_bound_violations,multiplier_bound_violations,non_negative_adapted_derivative_ratio,adapted_below_overstimulation_ratio,adapted_below_overstimulation_ratio_rapid_regime,max_sign_changes_1s
0,rapid,259.4,28.2354,136.551138,216.008,0.878485,0.8,1.1983,0.644589,0.88168,0.829379,1.0,0.792703,0,0,0,0.925175,0.819104,0.98282,2
1,still,246.2,27.1858,0.903158,9.326,1.200000,1.2,1.2000,0.819511,1.00000,0.000000,0.0,0.000000,0,0,0,1.000000,0.000000,NaN,0


In [11]:
still_mean = interpolate_mean_curves(still_runs)
rapid_mean = interpolate_mean_curves(rapid_runs)

still_mean.to_csv(OUTPUT_DIR / 'still_mean_curve.csv', index=False)
rapid_mean.to_csv(OUTPUT_DIR / 'rapid_mean_curve.csv', index=False)

make_mean_plot(still_mean, 'Still condition: mean active-phase response', OUTPUT_DIR / 'still_mean_plot.png')
make_mean_plot(rapid_mean, 'Rapid condition: mean active-phase response', OUTPUT_DIR / 'rapid_mean_plot.png')

print('Saved:')
print(OUTPUT_DIR / 'still_mean_plot.png')
print(OUTPUT_DIR / 'rapid_mean_plot.png')


Saved:
output/still_mean_plot.png
output/rapid_mean_plot.png


In [12]:
summary_text = f"""
RQ: Can a head-motion-based proxy mechanism produce a stable and bounded real-time adaptive intensity system on standalone VR hardware?

Results summary based on the active phase only (ActiveTriggers > 0):

H1 (Accumulation):
- In the still condition, the mean non-negative first-derivative ratio of Ladapted was {scenario_summary.loc[scenario_summary['scenario'] == 'still', 'non_negative_adapted_derivative_ratio'].iloc[0]:.3f}.
- Mean head rotation speed in the still condition was {scenario_summary.loc[scenario_summary['scenario'] == 'still', 'mean_head_rotation_speed'].iloc[0]:.2f} deg/s.
- Mean adaptation multiplier in the still condition was {scenario_summary.loc[scenario_summary['scenario'] == 'still', 'mean_adaptation_multiplier'].iloc[0]:.4f}.

H2 (Safety bounding):
- In the rapid condition, mean head rotation speed was {scenario_summary.loc[scenario_summary['scenario'] == 'rapid', 'mean_head_rotation_speed'].iloc[0]:.2f} deg/s.
- Mean adaptation multiplier in the rapid condition was {scenario_summary.loc[scenario_summary['scenario'] == 'rapid', 'mean_adaptation_multiplier'].iloc[0]:.4f}.
- Mean adapted-below-base ratio in the rapid regime was {scenario_summary.loc[scenario_summary['scenario'] == 'rapid', 'adapted_below_overstimulation_ratio_rapid_regime'].iloc[0]:.3f}.

H3 (Stability and boundedness):
- Overstimulation bound violations: {int(scenario_summary['overstimulation_bound_violations'].sum())}
- Adapted level bound violations: {int(scenario_summary['adapted_bound_violations'].sum())}
- Multiplier bound violations: {int(scenario_summary['multiplier_bound_violations'].sum())}
- Maximum observed sign changes in a 1-second window: {int(scenario_summary['max_sign_changes_1s'].max())}
"""

with open(OUTPUT_DIR / 'results_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary_text.strip())

print(summary_text)



RQ: Can a head-motion-based proxy mechanism produce a stable and bounded real-time adaptive intensity system on standalone VR hardware?

Results summary based on the active phase only (ActiveTriggers > 0):

H1 (Accumulation):
- In the still condition, the mean non-negative first-derivative ratio of Ladapted was 1.000.
- Mean head rotation speed in the still condition was 0.90 deg/s.
- Mean adaptation multiplier in the still condition was 1.2000.

H2 (Safety bounding):
- In the rapid condition, mean head rotation speed was 136.55 deg/s.
- Mean adaptation multiplier in the rapid condition was 0.8785.
- Mean adapted-below-base ratio in the rapid regime was 0.983.

H3 (Stability and boundedness):
- Overstimulation bound violations: 0
- Adapted level bound violations: 0
- Multiplier bound violations: 0
- Maximum observed sign changes in a 1-second window: 2



## Notebook outputs

After running all cells, the paper-ready artifacts will be saved in `analysis/output/`:

- `per_run_summary.csv`
- `scenario_summary.csv`
- `still_mean_curve.csv`
- `rapid_mean_curve.csv`
- `still_mean_plot.png`
- `rapid_mean_plot.png`
- `results_summary.txt`

Use `scenario_summary.csv` for your results table, the two PNG files for figures, and `results_summary.txt` as a draft for the Results section.
